# Etap EDA - krok 2: struktura i znaczenie pól

Cel: poznać pełny zestaw kolumn w `trip_updates`, ich typy i znaczenie, zanim przejdziemy
do szczegółowej oceny jakości konkretnych pól.

In [1]:
import duckdb
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW, LOGS, STATIC

print("RAW   :", RAW)
print("Czy RAW istnieje? ->", RAW.exists())

RAW   : C:\kmk-analiza\data\raw
Czy RAW istnieje? -> True


## Schemat kolumn

In [2]:
zapytanie = f"""
DESCRIBE
SELECT *
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet', hive_partitioning = true)
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

               column_name column_type null  key default extra
               observed_at      BIGINT  YES None    None  None
                 header_ts      BIGINT  YES None    None  None
                      feed     VARCHAR  YES None    None  None
                   trip_id     VARCHAR  YES None    None  None
                  route_id     VARCHAR  YES None    None  None
              direction_id      "NULL"  YES None    None  None
                start_date      "NULL"  YES None    None  None
trip_schedule_relationship      "NULL"  YES None    None  None
                vehicle_id     VARCHAR  YES None    None  None
             vehicle_label      "NULL"  YES None    None  None
             stop_sequence      "NULL"  YES None    None  None
                   stop_id     VARCHAR  YES None    None  None
             arrival_delay      DOUBLE  YES None    None  None
              arrival_time      DOUBLE  YES None    None  None
           departure_delay      DOUBLE  YES None    Non

**Identyfikatory kursu**: `trip_id`, `route_id`, `stop_id`, `vehicle_id` pozwalają złożyć kurs
i powiązać go z konkretnym przystankiem i pojazdem.

**Opóźnienie**: `arrival_delay`, `departure_delay` w sekundach, typ `DOUBLE`

**Czas**: `observed_at` (kiedy kolektor zaobserwował rekord), `header_ts` (znacznik z nagłówka
komunikatu źródła), `arrival_time`/`departure_time` (przewidywany/zrealizowany czas). Wszystko w UTC, później przekonwertujemy
na czas lokalny.

**Partycje**: `feed`, `date`, `hour` doklejone ze ścieżki dzięki `hive_partitioning`.

**Kilka kolumn ma typu `"NULL"`** (`direction_id`, `stop_sequence`, `trip_schedule_relationship`,
`vehicle_label`, `stop_schedule_relationship`) Oznacza to, że w pliku, z którego DuckDB wywnioskował
schemat, te kolumny były całkowicie puste. Nie wiadomo jeszcze, czy to dotyczy całego zbioru, czy
tylko tego jednego pliku.

## Niejednorodność schematu między plikami

Przy próbie policzenia wypełnienia kolumn oznaczonych jako `"NULL"` na całym zbiorze, DuckDB zwrócił
błąd. Nie mógł rzutować `BIGINT` na `"NULL"`, bo inny plik (feed M) miał `stop_sequence` zapisany jako
`BIGINT`, czyli wypełniony. To pokazuje, że **schemat różni się między plikami**, a DuckDB domyślnie
patrzy tylko na pierwszy napotkany plik.

**Rozwiązanie:** zawsze trzeba czytać `trip_updates` z `union_by_name = true`. Pozwala to DuckDB złożyć
schemat z wszystkich plików po nazwie kolumny, zamiast zgadywać z jednego.

## Wnioski

1. **Schemat `trip_updates` jest niejednorodny między plikami**: Kolektor czasami zapisuje pola
a czasem nie. DuckDB domyślnie bierze pod uwagę tylko pierwszy napotkany plik, co prowadzi do błędnego
typu `"NULL"` i błędów przy łączeniu wielu plików.
2. **Rozwiązanie**: zawsze używać `union_by_name = true` przy `read_parquet` na `trip_updates`.
3. **Zidentyfikowano kilka pól wymagających dalszej oceny**: Głównie `stop_sequence`.